# 🔬 RocksDB WAL Performance & Engineering Analysis
---
**Objective**: This notebook analyzes telemetry data extracted from the instrumented RocksDB core to quantify the overhead of WAL durability, recovery scaling, and internal fragmentation.

**Project Requirements Met**:
- ✅ Experimentation / Modification (Mandatory)
- ✅ Performance under different inputs
- ✅ Failure Analysis through data growth simulation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set Premium Aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Inter', 'Roboto', 'Arial']

try:
    df = pd.read_csv('../results/wal_performance_telemetry.csv')
    print(f"📡 Telemetry Ingested: {len(df)} events recorded.")
except Exception as e:
    print("⚠️ Data not yet generated. Please run 'wsl make run_all' in the experiments folder.")
    df = pd.DataFrame(columns=['Event', 'Count', 'Metric'])

df.head()

## 1. The Cost of Durability (WAL Throughput Analysis)
This experiment compares standard ingestion against WAL-disabled and Synced-WAL modes. 
**Instrumented Point**: `db/db_impl/db_impl_write.cc:2258`

In [ ]:
if not df.empty:
    batch = df[df['Event'] == 'WAL_Batch']
    if not batch.empty:
        plt.figure(figsize=(10, 6))
        ax = sns.barplot(x=['Vanilla (Buffered)', 'No-WAL', 'Strict Sync'], 
                         y=batch['Metric'].values[:3] / batch['Count'].values[:3] / 1000,
                         palette="flare")
        
        plt.title('Ingestion Latency Inside AddRecord()', fontsize=14, fontweight='bold')
        plt.ylabel('Microseconds per Latency (μs)', fontsize=12)
        plt.xlabel('WAL Configuration Mode', fontsize=12)
        
        for p in ax.patches:
            ax.annotate(f'{p.get_height():.2f}μs', 
                        (p.get_x() + p.get_width() / 2., p.get_height()), 
                        ha='center', va='center', 
                        xytext=(0, 9), 
                        textcoords='offset points', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig('../docs/images/exp1_latency.png')
        plt.show()

## 2. Recovery Scalability (Fault Tolerance MTTR)
Analyzes how replaying logs impacts system uptime after a failure. 
**Instrumented Point**: `db/db_impl/db_impl_open.cc:1132`

In [ ]:
if not df.empty:
    rec = df[df['Event'] == 'WAL_Recovery']
    if not rec.empty:
        plt.figure(figsize=(10, 6))
        sns.lineplot(x=rec['Count'], y=rec['Metric'], marker='o', markersize=8, 
                     linewidth=2.5, color='#34495e')
        
        plt.title('WAL Recovery Scaling (MTTR Analysis)', fontsize=14, fontweight='bold')
        plt.xlabel('Number of WAL Files to Replay', fontsize=12)
        plt.ylabel('Total Recovery Time (ms)', fontsize=12)
        
        plt.fill_between(rec['Count'], rec['Metric'], alpha=0.1, color='#34495e')
        plt.tight_layout()
        plt.savefig('../docs/images/exp2_recovery.png')
        plt.show()

## 3. Log Fragmentation & Header Overhead
Deep-dive into internal fragmentation when data exceeds the block boundaries.
**Instrumented Point**: `db/log_writer.cc:23-33`

In [ ]:
if not df.empty:
    overhead = df[df['Event'].isin(['WAL_Bytes_Payload', 'WAL_Bytes_Header'])]
    if not overhead.empty:
        counts = overhead.groupby('Event')['Metric'].sum()
        
        plt.figure(figsize=(8, 8))
        plt.pie(counts, labels=['Headers (Overhead)', 'Payload (Actual Data)'], 
                autopct='%1.1f%%', startangle=140, colors=['#e74c3c', '#2ecc71'],
                explode=(0.1, 0), shadow=True)
        
        plt.title('WAL Internal Efficiency Ratio', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig('../docs/images/exp5_efficiency.png')
        plt.show()